In [37]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [38]:
tickers = {
    #equities
    'SPY': 'S&P 500',
    'QQQ': 'Nasdaq 100',
    'IWM': 'Russell 2000',
    'EEM': 'Emerging Markets',

    #rates
    'SHY': '1-3Y Treasury',
    'IEF': '7-10Y Treasury',
    'TLT': '20Y+ Treasury',

    #credit
    'HYG': 'High Yield Credit',
    'LQD': 'Investment Grade Credit',

    #commodities
    'GLD': 'Gold',
    'SLV': 'Silver',
    'USO': 'Oil',
    'PDBC': 'Broad Commodities',

    #FX
    'UUP': 'US Dollar Index',
    'FXE': 'Euro',
    'FXY': 'Japanese Yen',
    'FXC': 'Canadian Dollar',

    #Sectors
    'XLF': 'Financials',
    'XLE': 'Energy',

    #Volatility
    'VIXY': 'Volatility Proxy'

}

#download 1 year of daily data for all tickers
data = yf.download(list(tickers.keys()), period = '1y', auto_adjust = True)

#take just closing prices
close_prices = data['Close']

#preview first 5 rows
close_prices.head()


[*********************100%***********************]  20 of 20 completed


Ticker,EEM,FXC,FXE,FXY,GLD,HYG,IEF,IWM,LQD,PDBC,QQQ,SHY,SLV,SPY,TLT,USO,UUP,VIXY,XLE,XLF
Date,,,,,,,,,,,,,,,,,,,,
2025-03-14,43.529343,67.801712,99.675911,62.060001,275.239990,74.421280,90.958519,200.633713,103.217377,12.822211,477.239624,79.391708,30.709999,556.277527,86.290520,72.279999,27.452431,50.380001,43.429363,47.853954
2025-03-17,44.271431,68.180275,100.033005,61.860001,276.730011,74.600548,91.035530,203.165237,103.427757,12.928101,480.333893,79.362808,30.809999,560.567200,86.721169,72.709999,27.346066,48.490002,44.121250,48.416126
2025-03-18,44.046852,68.120506,100.271057,61.820000,279.959991,74.468452,91.141434,201.401093,103.676353,12.899221,472.145477,79.362808,30.850000,554.508301,86.807281,72.209999,27.326725,49.830002,44.193821,48.356949
2025-03-19,44.154259,67.990990,99.814774,61.959999,281.109985,74.959053,91.420631,204.532959,104.288292,12.976232,478.463409,79.478409,30.750000,560.547424,87.257072,72.419998,27.394413,48.130001,44.914745,48.889542
2025-03-20,43.744160,68.020874,99.408089,61.970001,280.750000,74.855270,91.555420,203.234619,104.135307,12.995484,476.841675,79.497681,30.500000,558.926453,87.314476,73.730003,27.471771,47.330002,45.079250,48.958580


In [39]:
#calculate daily percentage returns for each asset
daily_returns = close_prices.pct_change()

#calculate cumulative returns (how much each asset has returned since day 1)
cumulative_returns = (1 + daily_returns).cumprod() - 1

#calculate 20-day rolling volatility (annualized)
rolling_vol = daily_returns.rolling(window = 20).std()*(252 ** 0.5)

#preview daily returns
daily_returns.head()

Ticker,EEM,FXC,FXE,FXY,GLD,HYG,IEF,IWM,LQD,PDBC,QQQ,SHY,SLV,SPY,TLT,USO,UUP,VIXY,XLE,XLF
Date,,,,,,,,,,,,,,,,,,,,
2025-03-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-17,0.017048,0.005583,0.003583,-0.003223,0.005414,0.002409,0.000847,0.012618,0.002038,0.008258,0.006484,-0.000364,0.003256,0.007711,0.004991,0.005949,-0.003875,-0.037515,0.015931,0.011748
2025-03-18,-0.005073,-0.000877,0.002380,-0.000647,0.011672,-0.001771,0.001163,-0.008683,0.002404,-0.002234,-0.017047,0.000000,0.001298,-0.010809,0.000993,-0.006877,-0.000707,0.027635,0.001645,-0.001222
2025-03-19,0.002438,-0.001901,-0.004551,0.002265,0.004108,0.006588,0.003063,0.015550,0.005902,0.005970,0.013381,0.001457,-0.003242,0.010891,0.005181,0.002908,0.002477,-0.034116,0.016313,0.011014
2025-03-20,-0.009288,0.000440,-0.004074,0.000161,-0.001281,-0.001385,0.001474,-0.006348,-0.001467,0.001484,-0.003389,0.000242,-0.008130,-0.002892,0.000658,0.018089,0.002824,-0.016622,0.003663,0.001412


In [40]:
#normalize prices to 100 so we can compare all assets on the same scale
normalized = (close_prices / close_prices.iloc[0]) * 100

#create the figure
fig = go.Figure()

#Add a line for each asset
for ticker, name in tickers.items():
    fig.add_trace(go.Scatter(
        x = normalized.index,
        y = normalized[ticker],
        name = name,
        mode = 'lines'
    ))

    #style the chart
    fig.update_layout (
        title = 'Cross-Asset Performance (Normalized to 100',
        xaxis_title = 'Date',
        yaxis_title = 'Normalized Price (Base = 100)',
        height = 600,
        hovermode = 'x unified',
        template = 'plotly_dark'
    )

fig.show()

In [41]:
#Grouping assets for cleaner charts
groups = {
    'Equities': ['SPY', 'QQQ', 'IWM', 'EEM'],
    'Rates': ['SHY', 'IEF', 'TLT'],
    'Credit': ['HYG', 'LQD'],
    'Commodities': ['GLD', 'SLV', 'USO', 'PDBC'],
    'FX': ['UUP', 'FXE', 'FXY', 'FXC'],
    'Sectors $ Vol': ['XLF', 'XLE', 'VIXY']
}

#Create a subplot for each group
fig = make_subplots(
    rows = 3, cols = 2,
    subplot_titles = list(groups.keys()),
    shared_xaxes = True
)

#Define row and column positions for each group
positions = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]

#loop through each group and add lines
for (row, col), (group_name, group_tickers) in zip(positions, groups.items()):
    for ticker in group_tickers:
        fig.add_trace(
            go.Scatter(
                x = normalized.index,
                y = normalized[ticker],
                name = tickers[ticker],
                mode = 'lines',
                legendgroup = group_name,
                showlegend = True
            ),
            row=row, col=col
        )

#style chart
fig.update_layout(
    height = 900,
    title = 'Cross-Asset Performace by Group (Normalized to 100)',
    template = 'plotly_dark',
    hovermode = 'x unified'
)

fig.show()

In [42]:
#Calculate the average normalized performance for each asset class group
group_performance = {}

for group_name, group_tickers in groups.items():
    #take the average of all assets in the group for each day
    group_performance[group_name] = normalized[group_tickers].mean(axis = 1)

#convert to a DataGrame
group_df = pd.DataFrame(group_performance)

#create chart
fig = go.Figure()

for group_name in group_df.columns:
    fig.add_trace(go.Scatter(
        x = group_df.index,
        y = group_df[group_name],
        name = group_name,
        mode = 'lines',
        line = dict (width = 2.5)
    ))

#style chart
fig.update_layout(
    title = 'Asset Class Performance Comparison (Group Averages, Normalized to 100)',
    xaxis_title = 'Date',
    yaxis_title = 'Normalized Price (Base = 100)',
    height = 600,
    hovermode = 'x unified',
    template = 'plotly_dark'
)

fig.show()

In [43]:
#create subplots for rolling volatility by group
fig = make_subplots(
    rows = 3, cols = 2,
    subplot_titles = list(groups.keys()),
    shared_xaxes = True
)

positions = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]

for (row, col), (group_name, group_tickers) in zip(positions, groups.items()):
    for ticker in group_tickers:
        fig.add_trace(
            go.Scatter(
                x = rolling_vol.index,
                y = rolling_vol[ticker]*100,        #converts to percentage
                name = tickers[ticker],
                mode = 'lines',
                legendgroup=group_name,
                showlegend=True
            ),
            row = row, col = col
        )

#customize chart
fig.update_layout(
    height = 900,
    title = '20-Day Rolling Volatility by Asset Class (%)',
    template = 'plotly_dark',
    hovermode = 'x unified'
)

fig.show()


In [44]:
#Calculate the corrlation matrix of daily returns
corr_matrix = daily_returns.corr()

#replace ticker symbols with full names
corr_matrix.columns = [tickers[t] for t in corr_matrix.columns]
corr_matrix.index = [tickers[t] for t in corr_matrix.index]

#create heatmap
fig = go.Figure(data = go.Heatmap(
    z = corr_matrix.values,
    x = corr_matrix.columns.tolist(),
    y = corr_matrix.index.tolist(),
    colorscale = 'RdBu',
    zmid = 0,
    zmin = -1,
    zmax = 1,
    text = corr_matrix.round(2).values,
    texttemplate='%{text}',
    textfont=dict(size=8),
    hoverongaps=False
))

fig.update_layout(
    title = 'Cross-Asset Correlation Matrix (Daily Returns)',
    height = 700,
    width = 900,
    template = 'plotly_dark',
    xaxis = dict(tickangle = 45)
)

fig.show()

In [45]:
#calculate each metric for every asset

#total return over period
total_return = (close_prices.iloc[-1] / close_prices.iloc[0] - 1) * 100

#current volatility (reccent 20-day rolling vol)
current_vol = rolling_vol.iloc[-1]*100

#sharpe ratio (annualized return/annualized volatility)
annualized_return = daily_returns.mean() * 252 * 100
annualized_vol = daily_returns.std() * (252 ** 0.5) * 100
sharpe_ratio = annualized_return / annualized_vol

#max drawdown (biggest peak to trough drop)
rolling_max = close_prices.cummax()
drawdown = (close_prices - rolling_max) / rolling_max * 100
max_drawdown = drawdown.min()

#build summary table
summary = pd.DataFrame({
    'Asset': [tickers[t] for t in close_prices.columns],
    'Total Return (%)': total_return.values.round(2),
    'Current Vol (%)': current_vol.values.round(2),
    'Sharpe Ratio': sharpe_ratio.values.round(2),
    'Max Drawdown (%)': max_drawdown.values.round(2)
})

#sort by total return with best performers at the top
summary = summary.sort_values('Total Return (%)', ascending = False).reset_index(drop = True)

summary

,Asset,Total Return (%),Current Vol (%),Sharpe Ratio,Max Drawdown (%)
0,Silver,136.70,66.41,1.87,-37.15
1,Gold,67.43,30.36,2.12,-13.87
2,Oil,65.87,62.40,1.57,-20.39
3,Broad Commodities,33.60,21.09,1.73,-11.07
4,Energy,32.86,16.83,1.28,-18.79
5,Emerging Markets,30.49,27.88,1.51,-15.04
6,Nasdaq 100,24.41,15.75,1.09,-15.69
7,Russell 2000,22.91,19.07,1.02,-16.30
8,S&P 500,19.06,11.55,1.02,-13.72
9,High Yield Credit,6.42,4.38,1.20,-4.19
